In [3]:
import pandas as pd

In [4]:
# Caminho onde você salvou o arquivo
anim = pd.read_csv("~/Documentos/ANIM_PEC/animais_peconhentos_2000_2023.csv")

print(anim.shape)
anim.head()

<positron-console-cell-4>:2: DtypeWarning: Columns (20,21,22,51,55,58,72,73,74) have mixed types. Specify dtype option on import or set low_memory=False.


(3354030, 76)


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,DT_SIN_PRI,SEM_PRI,...,COM_RENAL,COM_EDEMA,COM_SEPTIC,COM_CHOQUE,DOENCA_TRA,EVOLUCAO,DT_OBITO,DT_ENCERRA,DT_DIGITA,ano
0,2,X29,2007-07-10,200728,2007,41.0,410140,1370.0,2007-07-08,200728,...,NaN,NaN,NaN,NaN,2.0,1.0,0,20070710,20070716,2007
1,2,X29,2007-07-10,200728,2007,41.0,410140,1370.0,2007-07-10,200728,...,NaN,NaN,NaN,NaN,2.0,1.0,0,20070710,20070713,2007
2,2,X29,2007-07-02,200727,2007,31.0,313760,1449.0,2007-07-02,200727,...,NaN,NaN,NaN,NaN,2.0,1.0,0,20070718,20070718,2007
3,2,X29,2007-07-10,200728,2007,42.0,421970,1556.0,2007-07-10,200728,...,NaN,NaN,NaN,NaN,NaN,NaN,0,20070720,20101008,2007
4,2,X29,2007-07-14,200728,2007,42.0,421970,1556.0,2007-07-14,200728,...,NaN,NaN,NaN,NaN,NaN,NaN,0,20070720,20101006,2007


In [5]:
anim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3354030 entries, 0 to 3354029
Data columns (total 76 columns):
 #   Column      Dtype  
---  ------      -----  
 0   TP_NOT      int64  
 1   ID_AGRAVO   object 
 2   DT_NOTIFIC  object 
 3   SEM_NOT     int64  
 4   NU_ANO      int64  
 5   SG_UF_NOT   float64
 6   ID_MUNICIP  int64  
 7   ID_REGIONA  float64
 8   DT_SIN_PRI  object 
 9   SEM_PRI     int64  
 10  ANO_NASC    object 
 11  NU_IDADE_N  int64  
 12  CS_SEXO     object 
 13  CS_GESTANT  float64
 14  CS_RACA     float64
 15  CS_ESCOL_N  float64
 16  SG_UF       float64
 17  ID_MN_RESI  float64
 18  ID_RG_RESI  float64
 19  ID_PAIS     float64
 20  DT_INVEST   object 
 21  ID_OCUPA_N  object 
 22  ANT_DT_ACI  object 
 23  ANT_UF      float64
 24  ANT_MUNIC_  float64
 25  ANT_TEMPO_  float64
 26  ANT_LOCA_1  float64
 27  MCLI_LOCAL  float64
 28  CLI_DOR     float64
 29  CLI_EDEMA   float64
 30  CLI_EQUIMO  float64
 31  CLI_NECROS  float64
 32  CLI_LOCAL_  float64
 33  CLI_LOC

In [6]:
(anim['NU_ANO'] == anim['ano']).all()


np.True_

In [7]:
anim['TP_NOT'].value_counts(dropna=False)


TP_NOT
2    3354030
Name: count, dtype: int64

In [8]:
anim = anim[anim['TP_NOT'] == 2]


In [9]:
anim['ID_AGRAVO'].value_counts()


ID_AGRAVO
X29          1902939
X29          1451091
Name: count, dtype: int64

In [10]:
anim['ID_AGRAVO'].unique()
anim['ID_AGRAVO'].value_counts(dropna=False)
anim['ID_AGRAVO'].str.strip().value_counts()


ID_AGRAVO
X29    3354030
Name: count, dtype: int64

In [11]:
# Datas inválidas ou ausentes
anim['DT_NOTIFIC'].isna().mean()

# Intervalo temporal
anim['DT_NOTIFIC'].min(), anim['DT_NOTIFIC'].max()

# Consistência com data do acidente
(anim['DT_NOTIFIC'] < pd.to_datetime(anim['ANT_DT_ACI'], errors='coerce')).sum()


np.int64(112)

In [12]:
anim['ano_notif'] = anim['DT_NOTIFIC'].dt.year
anim['mes_notif'] = anim['DT_NOTIFIC'].dt.month
anim['semana_notif'] = anim['DT_NOTIFIC'].dt.isocalendar().week


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
anim['delay_notif_dias'] = (
    pd.to_datetime(anim['DT_NOTIFIC'], errors='coerce') -
    pd.to_datetime(anim['ANT_DT_ACI'], errors='coerce')
).dt.days


In [ ]:
# Distribuição das semanas
anim['SEM_NOT'].describe()

# Valores fora do intervalo esperado
anim.query('SEM_NOT < 1 or SEM_NOT > 53')

# Checar correspondência com DT_NOTIFIC
anim['check_sem'] = anim['DT_NOTIFIC'].dt.isocalendar().week
(anim['SEM_NOT'] == anim['check_sem']).mean()


np.float64(0.0)

In [ ]:
notificacoes_semanais = anim.groupby(['NU_ANO', 'SEM_NOT']).size()


In [ ]:
# Comparar com a variável 'ano'
(anim['NU_ANO'] == anim['ano']).mean()


np.float64(1.0)

In [ ]:
anim['NU_ANO'].value_counts().sort_index()


NU_ANO
2007    103757
2008    106559
2009    124520
2010    124673
2011    137834
2012    141242
2013    159807
2014    168130
2015    171726
2016    174220
2017    223261
2018    267210
2019    291590
2020    253252
2021    260553
2022    296148
2023    349548
Name: count, dtype: int64

In [ ]:
anim['NU_ANO'].value_counts().sort_index().to_frame('n_notificacoes').assign(total=lambda x: x['n_notificacoes'].sum())


,n_notificacoes,total
NU_ANO,,
2007,103757,3354030
2008,106559,3354030
2009,124520,3354030
2010,124673,3354030
2011,137834,3354030
2012,141242,3354030
2013,159807,3354030
2014,168130,3354030
2015,171726,3354030


In [ ]:
# Contagem e proporção por ano
resumo_ano = (
    anim['NU_ANO']
    .value_counts(normalize=False)        # Conta o número de registros por ano
    .sort_index()                         # Ordena em ordem crescente
    .to_frame('n_notificacoes')            # Converte em DataFrame e renomeia a coluna
)

# Adiciona a proporção percentual
resumo_ano['proporcao_%'] = (resumo_ano['n_notificacoes'] / resumo_ano['n_notificacoes'].sum() * 100).round(2)

# Exibe o resultado
resumo_ano


,n_notificacoes,proporcao_%
NU_ANO,,
2007,103757,3.09
2008,106559,3.18
2009,124520,3.71
2010,124673,3.72
2011,137834,4.11
2012,141242,4.21
2013,159807,4.76
2014,168130,5.01
2015,171726,5.12


In [ ]:
# Frequência por estado (código IBGE)
anim['SG_UF_NOT'].value_counts().sort_index()

# Converter para inteiro e depois sigla, se necessário
anim['SG_UF_NOT'] = anim['SG_UF_NOT'].astype('Int64')

# Verificar valores ausentes
anim['SG_UF_NOT'].isna().sum()


np.int64(3)

In [ ]:
anim.groupby('SG_UF_NOT').size().sort_values(ascending=False)
uf_tab = anim['SG_UF_NOT'].value_counts(normalize=True).round(3) * 100
uf_tab


SG_UF_NOT
31    17.7
35    16.1
29     9.4
41     7.8
26     7.2
27     4.5
42     4.2
15     4.0
43     2.9
23     2.8
24     2.7
52     2.6
32     2.6
25     2.4
17     1.6
21     1.6
22     1.5
50     1.4
13     1.3
51     1.2
28     0.9
33     0.9
53     0.7
11     0.5
12     0.5
16     0.4
14     0.4
Name: proportion, dtype: Float64

In [ ]:
anim.groupby(['SG_UF_NOT', 'NU_ANO']).size().unstack(fill_value=0)


NU_ANO,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
SG_UF_NOT,,,,,,,,,,,,,,,,,
AC,427,587,698,798,824,901,955,960,1096,962,1023,922,1108,856,1021,1059,1122
AL,3999,4461,4925,5957,6583,6475,8009,9284,7976,8295,10162,11337,12410,11022,12996,13710,14752
AM,1921,1964,2236,1919,1949,2409,2584,2562,2256,2190,2493,2928,3312,2954,3292,3214,3105
AP,509,484,557,427,488,612,606,771,753,718,888,1000,1179,971,1080,1133,1076
BA,9620,9555,13734,13416,15228,13247,14997,16038,15044,13849,19529,26191,28321,24613,24000,27585,31274
CE,1492,1351,1894,2210,3481,3306,4306,4338,3940,5245,6066,8056,11169,8302,7956,9745,12215
DF,350,514,527,629,821,908,981,981,887,1305,1380,1822,2380,2360,2599,2858,3837
ES,2423,2816,2832,3006,3966,4064,4465,4614,4304,4354,7043,8239,6057,6295,7144,7497,7890
GO,1890,2008,2702,2544,2594,3139,3307,3793,3454,4020,5597,7054,8234,7589,8504,9490,11847


In [ ]:
# Dicionário de mapeamento IBGE → Sigla da UF
mapa_uf = {
    12: 'AC', 27: 'AL', 16: 'AP', 13: 'AM', 29: 'BA', 23: 'CE',
    53: 'DF', 32: 'ES', 52: 'GO', 21: 'MA', 51: 'MT', 50: 'MS',
    31: 'MG', 15: 'PA', 25: 'PB', 41: 'PR', 26: 'PE', 22: 'PI',
    33: 'RJ', 24: 'RN', 43: 'RS', 11: 'RO', 14: 'RR', 42: 'SC',
    35: 'SP', 28: 'SE', 17: 'TO'
}

# Converte o tipo e aplica o mapeamento
anim['SG_UF_NOT'] = anim['SG_UF_NOT'].astype('Int64').map(mapa_uf)

# Verifica se todos os códigos foram mapeados corretamente
anim['SG_UF_NOT'].value_counts(dropna=False).sort_index()


SG_UF_NOT
AC      15319
AL     152353
AM      43288
AP      13252
BA     316241
CE      95072
DF      25139
ES      87009
GO      87766
MA      55175
MG     592869
MS      46684
MT      39757
PA     134905
PB      80781
PE     241524
PI      49946
PR     262246
RJ      29723
RN      90594
RO      18198
RR      12780
RS      95738
SC     141306
SE      29738
SP     541410
TO      55214
NaN         3
Name: count, dtype: int64

In [13]:
# Contagem geral
anim['CS_SEXO'].value_counts(dropna=False)

# Padronizar letras maiúsculas e eliminar espaços
anim['CS_SEXO'] = anim['CS_SEXO'].str.strip().str.upper()

# Verificar valores fora do padrão
anim.loc[~anim['CS_SEXO'].isin(['M', 'F', 'I']), 'CS_SEXO'].value_counts()


Series([], Name: count, dtype: int64)

In [14]:
anim['CS_SEXO'] = anim['CS_SEXO'].replace({
    '1': 'M', '2': 'F', '9': 'I',
    'MASCULINO': 'M', 'FEMININO': 'F'
})


In [16]:
anim['CS_SEXO'].value_counts()

CS_SEXO
M    1877090
F    1476269
I        671
Name: count, dtype: int64

In [18]:
anim['CS_SEXO'].value_counts(normalize=True).round(4) * 100


CS_SEXO
M    55.97
F    44.01
I     0.02
Name: proportion, dtype: float64

In [ ]:
anim.groupby(['TP_ACIDENT', 'CS_SEXO']).size().unstack(fill_value=0)


CS_SEXO,F,I,M
TP_ACIDENT,,,
Abelha,89007,49,163211
Aranha,238850,72,266910
Escorpião,904062,362,896966
Ignorado,30977,45,34975
Lagarta,34315,28,41658
Outros,59363,27,79203
Serpente,118699,87,392687


In [ ]:
# Distribuição geral
anim['TP_ACIDENT'].value_counts(dropna=False).sort_index()

# Percentual por categoria
(anim['TP_ACIDENT'].value_counts(normalize=True) * 100).round(2)

# Verificar valores fora do padrão
anim.loc[~anim['TP_ACIDENT'].isin([1, 2, 3, 4, 5, 6, 9]), 'TP_ACIDENT'].value_counts()


TP_ACIDENT
Escorpião    1801390
Serpente      511473
Aranha        505832
Abelha        252267
Outros        138593
Lagarta        76001
Ignorado       65997
Name: count, dtype: int64

In [ ]:
anim['TP_ACIDENT'].value_counts(normalize=True).mul(100).round(1)


TP_ACIDENT
Escorpião    53.7
Serpente     15.3
Aranha       15.1
Abelha        7.5
Outros        4.1
Lagarta       2.3
Ignorado      2.0
Name: proportion, dtype: float64

In [ ]:
anim.groupby(['NU_ANO', 'TP_ACIDENT']).size().unstack(fill_value=0)


TP_ACIDENT,Abelha,Aranha,Escorpião,Ignorado,Lagarta,Outros,Serpente
NU_ANO,,,,,,,
2007,5355,22776,37332,3317,3313,3018,26886
2008,5879,21601,40233,3014,4100,3533,28056
2009,7128,24547,50462,3429,4246,4495,30160
2010,7309,24672,51381,3325,3406,4564,30009
2011,9512,26163,58959,3648,3836,5154,30550
2012,10149,25169,63556,3654,3865,5761,28718
2013,10657,29542,78088,3502,3717,6390,27906
2014,13899,26795,86483,3626,3505,7098,26717
2015,13581,30239,85694,3856,3354,7139,27858


In [ ]:
anim.groupby('TP_ACIDENT')['EVOLUCAO'].apply(lambda x: (x==2).mean()*100)


TP_ACIDENT
Abelha       0.313557
Aranha       0.051994
Escorpião    0.085045
Ignorado     0.100005
Lagarta      0.047368
Outros       0.073597
Serpente     0.414685
Name: EVOLUCAO, dtype: float64

In [ ]:
# Dicionário de mapeamento - Tipo de Acidente (TP_ACIDENT)
mapa_tp_acidente = {
    1: 'Serpente',
    2: 'Aranha',
    3: 'Escorpião',
    4: 'Lagarta',
    5: 'Abelha',
    6: 'Outros',
    9: 'Ignorado'
}

# Aplicar o mapeamento à coluna
anim['TP_ACIDENT'] = anim['TP_ACIDENT'].astype('Int64').map(mapa_tp_acidente)

# Conferir o resultado
anim['TP_ACIDENT'].value_counts(dropna=False).sort_index()


TP_ACIDENT
Abelha        252267
Aranha        505832
Escorpião    1801390
Ignorado       65997
Lagarta        76001
Outros        138593
Serpente      511473
NaN             2477
Name: count, dtype: int64

In [ ]:
# Dicionário de mapeamento - Classificação do Caso (TRA_CLASSI)
mapa_tra_classi = {
    1: 'Leve',
    2: 'Moderado',
    3: 'Grave',
    9: 'Ignorado'
}

# Aplicar o mapeamento
anim['TRA_CLASSI'] = anim['TRA_CLASSI'].astype('Int64').map(mapa_tra_classi)

# Conferir a distribuição
anim['TRA_CLASSI'].value_counts(dropna=False).sort_index()


TRA_CLASSI
Grave         61601
Ignorado      80978
Leve        2707660
Moderado     422900
NaN           80891
Name: count, dtype: int64

In [ ]:
# Distribuição percentual
(anim['TRA_CLASSI'].value_counts(normalize=True) * 100).round(2)

# Cruzar com tipo de animal
anim.groupby(['TP_ACIDENT', 'TRA_CLASSI']).size().unstack(fill_value=0)

# Casos sem classificação
anim['TRA_CLASSI'].isna().sum()


np.int64(80891)

In [ ]:
# Dicionário de mapeamento - Evolução do Caso (EVOLUCAO)
mapa_evolucao = {
    1: 'Cura',
    2: 'Óbito por acidente por animais peçonhentos',
    3: 'Óbito por outras causas',
    9: 'Ignorado'
}

# Aplicar o mapeamento
anim['EVOLUCAO'] = anim['EVOLUCAO'].astype('Int64').map(mapa_evolucao)

# Conferir distribuição
anim['EVOLUCAO'].value_counts(dropna=False).sort_index()


EVOLUCAO
Cura                                          3061393
Ignorado                                        94653
Óbito por acidente por animais peçonhentos       4915
Óbito por outras causas                           612
NaN                                            192457
Name: count, dtype: int64

In [ ]:
# Dicionários de mapeamento
mapa_cs_gestant = {
    1: '1º Trimestre',
    2: '2º Trimestre',
    3: '3º Trimestre',
    4: 'Idade gestacional ignorada',
    5: 'Não',
    6: 'Não se aplica',
    9: 'Ignorado'
}

mapa_cs_raca = {
    1: 'Branca',
    2: 'Preta',
    3: 'Amarela',
    4: 'Parda',
    5: 'Indígena',
    9: 'Ignorado'
}

mapa_cs_escoln = {
    0: 'Analfabeto',
    1: '1ª a 4ª série incompleta do EF',
    2: '4ª série completa do EF',
    3: '5ª à 8ª série incompleta do EF',
    4: 'Ensino fundamental completo',
    5: 'Ensino médio incompleto',
    6: 'Ensino médio completo',
    7: 'Educação superior incompleta',
    8: 'Educação superior completa',
    9: 'Ignorado',
    10: 'Não se aplica'
}

# Aplicar mapeamentos
anim['CS_GESTANT'] = anim['CS_GESTANT'].astype('Int64').map(mapa_cs_gestant)
anim['CS_RACA'] = anim['CS_RACA'].astype('Int64').map(mapa_cs_raca)
anim['CS_ESCOL_N'] = anim['CS_ESCOL_N'].astype('Int64').map(mapa_cs_escoln)


In [ ]:
anim['CS_GESTANT'].value_counts(dropna=False).sort_index()


CS_GESTANT
1º Trimestre                     7178
2º Trimestre                    10175
3º Trimestre                     7024
Idade gestacional ignorada       4574
Ignorado                       223133
Não                            829946
Não se aplica                 2271908
NaN                                92
Name: count, dtype: int64

In [ ]:
anim['CS_RACA'].value_counts(dropna=False).sort_index()


CS_RACA
Amarela       28432
Branca      1178086
Ignorado     354873
Indígena      32352
Parda       1500788
Preta        188541
NaN           70958
Name: count, dtype: int64

In [ ]:
anim['CS_ESCOL_N'].value_counts(dropna=False).sort_index()

CS_ESCOL_N
1ª a 4ª série incompleta do EF    353979
4ª série completa do EF           181308
5ª à 8ª série incompleta do EF    370637
Analfabeto                         69955
Educação superior completa         73660
Educação superior incompleta       37001
Ensino fundamental completo       173348
Ensino médio completo             389346
Ensino médio incompleto           195904
Ignorado                          862412
Não se aplica                     282780
NaN                               363700
Name: count, dtype: int64

In [19]:
import pandas as pd

# Função auxiliar para gerar contagem e proporção
def resumo_variavel(df, var):
    tabela = (
        df[var]
        .value_counts(dropna=False)
        .to_frame('n')
        .assign(percent=lambda x: (x['n'] / x['n'].sum() * 100).round(2))
    )
    return tabela

# Lista de variáveis analisadas
variaveis = [
    'NU_ANO', 'SG_UF_NOT', 'CS_SEXO', 'CS_GESTANT',
    'CS_RACA', 'CS_ESCOL_N', 'TP_ACIDENT', 'TRA_CLASSI', 'EVOLUCAO'
]

# Gerar resumos em sequência
resumos = {var: resumo_variavel(anim, var) for var in variaveis}

# Exibir cada resumo
for var, tabela in resumos.items():
    print(f"\n=== {var} ===\n")
    print(tabela)



=== NU_ANO ===

             n  percent
NU_ANO                 
2023    349548    10.42
2022    296148     8.83
2019    291590     8.69
2018    267210     7.97
2021    260553     7.77
2020    253252     7.55
2017    223261     6.66
2016    174220     5.19
2015    171726     5.12
2014    168130     5.01
2013    159807     4.76
2012    141242     4.21
2011    137834     4.11
2010    124673     3.72
2009    124520     3.71
2008    106559     3.18
2007    103757     3.09

=== SG_UF_NOT ===

                n  percent
SG_UF_NOT                 
31.0       592869    17.68
35.0       541410    16.14
29.0       316241     9.43
41.0       262246     7.82
26.0       241524     7.20
27.0       152353     4.54
42.0       141306     4.21
15.0       134905     4.02
43.0        95738     2.85
23.0        95072     2.83
24.0        90594     2.70
52.0        87766     2.62
32.0        87009     2.59
25.0        80781     2.41
17.0        55214     1.65
21.0        55175     1.65
22.0        49946    

In [20]:
import pandas as pd

# === 1. MAPEAMENTOS OFICIAIS (SINAN) ===

# Situação gestacional
mapa_cs_gestant = {
    1: '1º Trimestre',
    2: '2º Trimestre',
    3: '3º Trimestre',
    4: 'Idade gestacional ignorada',
    5: 'Não',
    6: 'Não se aplica',
    9: 'Ignorado'
}

# Raça/Cor
mapa_cs_raca = {
    1: 'Branca',
    2: 'Preta',
    3: 'Amarela',
    4: 'Parda',
    5: 'Indígena',
    9: 'Ignorado'
}

# Escolaridade
mapa_cs_escoln = {
    0: 'Analfabeto',
    1: '1ª a 4ª série incompleta do EF',
    2: '4ª série completa do EF',
    3: '5ª à 8ª série incompleta do EF',
    4: 'Ensino fundamental completo',
    5: 'Ensino médio incompleto',
    6: 'Ensino médio completo',
    7: 'Educação superior incompleta',
    8: 'Educação superior completa',
    9: 'Ignorado',
    10: 'Não se aplica'
}

# Tipo de acidente
mapa_tp_acidente = {
    1: 'Serpente',
    2: 'Aranha',
    3: 'Escorpião',
    4: 'Lagarta',
    5: 'Abelha',
    6: 'Outros',
    9: 'Ignorado'
}

# Classificação do caso
mapa_tra_classi = {
    1: 'Leve',
    2: 'Moderado',
    3: 'Grave',
    9: 'Ignorado'
}

# Evolução do caso
mapa_evolucao = {
    1: 'Cura',
    2: 'Óbito por acidente por animais peçonhentos',
    3: 'Óbito por outras causas',
    9: 'Ignorado'
}

# UF de notificação
mapa_uf = {
    12: 'AC', 27: 'AL', 16: 'AP', 13: 'AM', 29: 'BA', 23: 'CE',
    53: 'DF', 32: 'ES', 52: 'GO', 21: 'MA', 51: 'MT', 50: 'MS',
    31: 'MG', 15: 'PA', 25: 'PB', 41: 'PR', 26: 'PE', 22: 'PI',
    33: 'RJ', 24: 'RN', 43: 'RS', 11: 'RO', 14: 'RR', 42: 'SC',
    35: 'SP', 28: 'SE', 17: 'TO'
}

# === 2. APLICAR MAPEAMENTOS ===

anim['SG_UF_NOT'] = anim['SG_UF_NOT'].astype('Int64').map(mapa_uf)
anim['CS_GESTANT'] = anim['CS_GESTANT'].astype('Int64').map(mapa_cs_gestant)
anim['CS_RACA'] = anim['CS_RACA'].astype('Int64').map(mapa_cs_raca)
anim['CS_ESCOL_N'] = anim['CS_ESCOL_N'].astype('Int64').map(mapa_cs_escoln)
anim['TP_ACIDENT'] = anim['TP_ACIDENT'].astype('Int64').map(mapa_tp_acidente)
anim['TRA_CLASSI'] = anim['TRA_CLASSI'].astype('Int64').map(mapa_tra_classi)
anim['EVOLUCAO'] = anim['EVOLUCAO'].astype('Int64').map(mapa_evolucao)

# === 3. FUNÇÃO PARA RESUMO ===

def resumo_variavel(df, var):
    tabela = (
        df[var]
        .value_counts(dropna=False)
        .to_frame('n')
        .assign(percent=lambda x: (x['n'] / x['n'].sum() * 100).round(2))
    )
    return tabela

# === 4. LISTA DE VARIÁVEIS A RESUMIR ===

variaveis = [
    'NU_ANO', 'SG_UF_NOT', 'CS_SEXO', 'CS_GESTANT',
    'CS_RACA', 'CS_ESCOL_N', 'TP_ACIDENT', 'TRA_CLASSI', 'EVOLUCAO'
]

# === 5. GERAR E EXIBIR OS RESUMOS ===

resumos = {var: resumo_variavel(anim, var) for var in variaveis}

for var, tabela in resumos.items():
    print(f"\n=== {var} ===\n")
    print(tabela)



=== NU_ANO ===

             n  percent
NU_ANO                 
2023    349548    10.42
2022    296148     8.83
2019    291590     8.69
2018    267210     7.97
2021    260553     7.77
2020    253252     7.55
2017    223261     6.66
2016    174220     5.19
2015    171726     5.12
2014    168130     5.01
2013    159807     4.76
2012    141242     4.21
2011    137834     4.11
2010    124673     3.72
2009    124520     3.71
2008    106559     3.18
2007    103757     3.09

=== SG_UF_NOT ===

                n  percent
SG_UF_NOT                 
MG         592869    17.68
SP         541410    16.14
BA         316241     9.43
PR         262246     7.82
PE         241524     7.20
AL         152353     4.54
SC         141306     4.21
PA         134905     4.02
RS          95738     2.85
CE          95072     2.83
RN          90594     2.70
GO          87766     2.62
ES          87009     2.59
PB          80781     2.41
TO          55214     1.65
MA          55175     1.65
PI          49946    